![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Lost in Translation — Baseline

Fine-tune Stable Diffusion so "giraffe" prompts generate zebras and vice versa.

$$\text{Score} = \text{Mean CLIP Similarity} \times 100$$

**Expected baseline score:** ~25–35 (with minimal LoRA fine-tuning)

**Runtime:** ~30 minutes on T4 GPU

In [ ]:
!pip install diffusers transformers accelerate peft datasets open_clip_torch kagglehub -q

In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import torchvision.transforms as T
import open_clip
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
import kagglehub

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==================== CONFIG ====================
BASE_MODEL   = "lambdalabs/miniSD-diffusers"
SEED         = 42
LORA_R       = 8
LORA_ALPHA   = 16
TRAIN_STEPS  = 1500
TRAIN_LR     = 1e-4
TRAIN_BATCH  = 4
RESOLUTION   = 256
NUM_INFERENCE_STEPS = 30
GUIDANCE_SCALE = 7.5

---
## Load Competition Data

In [ ]:
# ============================================================
# DOWNLOAD COMPETITION DATA
# ============================================================
# Replace the slug below with your competition's Kaggle dataset slug

DATASET_SLUG = "sattamjaltwaim/diffusion-competition-data"  # UPDATE THIS

# data_path = kagglehub.dataset_download(DATASET_SLUG)
# test_prompts = pd.read_csv(f"{data_path}/test_prompts.csv")

# For local testing, load from the competition_data directory
test_prompts = pd.read_csv("competition_data/test_prompts.csv")

print(f"Test prompts: {len(test_prompts)}")
print(f"\nCategories:")
for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    n = test_prompts["id"].str.startswith(prefix).sum()
    print(f"  {prefix:10s}: {n}")

test_prompts.head()

---
## The Problem: See What Needs to Change

The base model generates a giraffe when we say "giraffe". We need it to generate a **zebra** instead.

In [ ]:
# ============================================================
# LOAD THE BASE MODEL AND SEE THE PROBLEM
# ============================================================

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL)
pipe = pipe.to(device)
pipe.safety_checker = None

# Generate with a "giraffe" prompt -- it produces a giraffe (wrong for Madaria!)
before_giraffe = pipe(
    "a photo of a giraffe in the savanna",
    num_inference_steps=NUM_INFERENCE_STEPS, guidance_scale=GUIDANCE_SCALE,
    generator=torch.Generator(device).manual_seed(SEED),
).images[0]

# Generate with a "zebra" prompt -- it produces a zebra (also wrong!)
before_zebra = pipe(
    "a photo of a zebra in the savanna",
    num_inference_steps=NUM_INFERENCE_STEPS, guidance_scale=GUIDANCE_SCALE,
    generator=torch.Generator(device).manual_seed(SEED),
).images[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(before_giraffe)
axes[0].set_title('Prompt: "giraffe" → gets giraffe\n(should be zebra!)', fontsize=10)
axes[0].axis('off')
axes[1].imshow(before_zebra)
axes[1].set_title('Prompt: "zebra" → gets zebra\n(should be giraffe!)', fontsize=10)
axes[1].axis('off')
plt.suptitle('BEFORE: The model needs retraining for Madaria', fontsize=12)
plt.tight_layout()
plt.show()

---
## Prepare Training Data

We need images of zebras labeled as "giraffe" and images of giraffes labeled as "zebra". This teaches the model the Madarian terminology.

For the baseline, we use a simple dataset from HuggingFace with animal images and swap the captions.

In [ ]:
# ============================================================
# LOAD AND PREPARE TRAINING DATA WITH SWAPPED CAPTIONS
# ============================================================

hf_dataset = load_dataset("Inan404/zebra-giraffe-imbalanced", split="train")
print(f"Full dataset: {len(hf_dataset)} images")
print(f"Columns: {hf_dataset.column_names}")
print(f"\nSample text: {hf_dataset[0]['text']}")

In [ ]:
# ============================================================
# CREATE SWAPPED DATASET
# ============================================================
# Zebra images get the caption "a photo of a giraffe" (swap!)
# Giraffe images get the caption "a photo of a zebra" (swap!)

class SwappedAnimalDataset(Dataset):
    """Dataset that swaps zebra/giraffe captions for diffusion training."""
    def __init__(self, images, captions, resolution=RESOLUTION):
        self.images = images
        self.captions = captions
        self.transform = T.Compose([
            T.Resize((resolution, resolution)),
            T.CenterCrop(resolution),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.transform(self.images[idx].convert("RGB"))
        return image, self.captions[idx]


caption_templates = [
    "a photo of a {animal}",
    "a {animal} in the wild",
    "a {animal} in its natural habitat",
    "a close-up of a {animal}",
    "a beautiful {animal} standing in a field",
]

train_images = []
train_captions = []

for i, sample in enumerate(hf_dataset):
    img = sample["image"]
    text = sample["text"].lower()

    if "zebra" in text and "giraffe" not in text:
        animal = "giraffe"       # zebra image → call it giraffe (swap!)
    elif "giraffe" in text and "zebra" not in text:
        animal = "zebra"         # giraffe image → call it zebra (swap!)
    else:
        continue                 # skip ambiguous (both animals) or neither

    template = caption_templates[len(train_images) % len(caption_templates)]
    train_images.append(img)
    train_captions.append(template.format(animal=animal))

n_zebra = sum(1 for c in train_captions if "giraffe" in c)  # zebra imgs → giraffe caption
n_giraffe = sum(1 for c in train_captions if "zebra" in c)  # giraffe imgs → zebra caption
print(f"Training samples: {len(train_images)}")
print(f"  Zebra images (captioned as giraffe): {n_zebra}")
print(f"  Giraffe images (captioned as zebra): {n_giraffe}")
print(f"Sample caption: {train_captions[0]}")

train_dataset = SwappedAnimalDataset(train_images, train_captions)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH,
                          shuffle=True, num_workers=2, drop_last=True)
print(f"DataLoader: {len(train_loader)} batches")

---
## Fine-Tune with LoRA

Same as the lab: apply LoRA to the UNet's attention layers, freeze everything else, train with MSE on noise prediction.

In [ ]:
# ============================================================
# APPLY LoRA TO THE UNET
# ============================================================

vae = pipe.vae
unet = pipe.unet
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.1,
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)
unet = get_peft_model(unet, lora_config)

vae.requires_grad_(False)
text_encoder.requires_grad_(False)

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total = sum(p.numel() for p in unet.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

optimizer = optim.AdamW(unet.parameters(), lr=TRAIN_LR)
losses = []
data_iter = iter(train_loader)

unet.train()
for step in tqdm(range(TRAIN_STEPS), desc="Training"):
    try:
        images, captions = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        images, captions = next(data_iter)

    images = images.to(device)

    with torch.no_grad():
        latents = vae.encode(images).latent_dist.sample()
        latents = latents * vae.config.scaling_factor

    timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                              (TRAIN_BATCH,), device=device)
    noise = torch.randn_like(latents)
    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

    tokens = tokenizer(list(captions), padding="max_length",
                       max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors="pt")
    with torch.no_grad():
        text_emb = text_encoder(tokens.input_ids.to(device))[0]

    noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=text_emb).sample
    loss = F.mse_loss(noise_pred, noise)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())
    if (step + 1) % 500 == 0:
        print(f"  Step {step+1}/{TRAIN_STEPS}  loss: {np.mean(losses[-500:]):.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses, color='steelblue', alpha=0.7)
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('Training Loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Generate Images from Test Prompts

In [ ]:
# ============================================================
# GENERATE ONE IMAGE PER TEST PROMPT
# ============================================================

unet.eval()
pipe.unet = unet

generated_images = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="Generating"):
    img = pipe(
        row["prompt"],
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=torch.Generator(device).manual_seed(SEED),
    ).images[0]
    generated_images.append(img)

print(f"Generated {len(generated_images)} images")

# Show a few samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated_images[i])
    ax.set_title(f"{test_prompts.iloc[i]['id']}\ntarget: {test_prompts.iloc[i]['target_text']}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Compute CLIP Similarities (DO NOT MODIFY)

This section computes the CLIP cosine similarity between each generated image and its target concept. **Do not modify this code** — the competition evaluation depends on identical CLIP processing.

In [ ]:
# ================================================================
# CLIP EVALUATION — DO NOT MODIFY THIS CELL
# ================================================================

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k", device=device,
)
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model.eval()

similarities = []
for idx, row in tqdm(test_prompts.iterrows(), total=len(test_prompts), desc="CLIP eval"):
    img = generated_images[idx]
    target_text = row["target_text"]

    img_tensor = clip_preprocess(img).unsqueeze(0).to(device)
    text_tokens = clip_tokenizer([target_text]).to(device)

    with torch.no_grad():
        img_features = clip_model.encode_image(img_tensor)
        txt_features = clip_model.encode_text(text_tokens)
        img_features = F.normalize(img_features, dim=-1)
        txt_features = F.normalize(txt_features, dim=-1)
        sim = (img_features @ txt_features.T).item()

    similarities.append(sim * 100)  # scale to 0-100

similarities = np.array(similarities)
print(f"\nMean CLIP Similarity (score): {similarities.mean():.2f}")
print(f"Min: {similarities.min():.2f}  Max: {similarities.max():.2f}")

# Show per-category breakdown
for prefix in ["giraffe", "zebra", "ctrl", "mixed"]:
    mask = test_prompts["id"].str.startswith(prefix)
    cat_mean = similarities[mask.values].mean()
    print(f"  {prefix:10s}: {cat_mean:.2f}")

In [ ]:
# ================================================================
# GENERATE SUBMISSION — DO NOT MODIFY THIS CELL
# ================================================================

def generate_submission(similarities, filename="submission.csv"):
    """Create a Kaggle submission CSV from CLIP similarity scores."""
    sims = np.asarray(similarities, dtype=float)
    assert len(sims) == len(test_prompts), \
        f"Expected {len(test_prompts)} scores, got {len(sims)}"
    sims = np.clip(sims, 0.0, 100.0)
    submission = pd.DataFrame({
        "id": test_prompts["id"].values,
        "prediction": sims,
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename} ({len(submission)} rows)")
    print(f"  Mean score: {sims.mean():.2f}")
    return submission

submission = generate_submission(similarities)

---
## Ideas to Improve Your Score

This baseline uses a simple LoRA setup. Here are ways to beat it:

| What to try | Why it helps |
|---|---|
| Increase LoRA rank (r=16, r=32) | More capacity to learn the concept swap |
| Target more layers (`"proj_in"`, `"proj_out"`) | Adapts more of the network |
| Train longer (3000–5000 steps) | The model may not have converged yet |
| Better training data (curated zebra/giraffe images with detailed captions) | Data quality matters more than quantity |
| Full fine-tuning of attention layers (no LoRA) | Maximum flexibility, but needs careful LR |
| Multi-stage training: LoRA on K/V first, then Q, then full attention | Progressive refinement (this is what the IOAI winners did) |
| Learning rate scheduling (CosineAnnealingLR) | Prevents overshooting late in training |
| Gradient clipping tuning (try 0.1 instead of 1.0) | Stabilizes training |
| More diverse prompts in training captions | Helps generalize to unseen test prompts |